# RAG 检索问答演示

本 Notebook 演示完整的 RAG（Retrieval-Augmented Generation）流程：

```
文档加载 → 文本切分 → Embedding 向量化 → FAISS 相似度检索 → LLM 生成答案
```

所有 Embedding 和 LLM 调用均使用智谱 AI API，配置从 `config.json` 读取。

## 0. 读取配置文件

配置信息从当前目录下的 `config.json` 读取，包含智谱 API Key、聊天模型和 Embedding 模型。

In [1]:
import json
import os

CONFIG_FILE = "config.json"

config = {
    "zhipu_api_key": "your-zhipu-api-key-here",
    "chat_model": "glm-4-flash",
    "embedding_model": "embedding-3"
}

if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE, "r", encoding="utf-8") as f:
        config.update(json.load(f))

ZHIPU_API_KEY = config["zhipu_api_key"]
CHAT_MODEL = config["chat_model"]
EMBEDDING_MODEL = config["embedding_model"]

print(f"智谱 API Key 是否已设置: {bool(ZHIPU_API_KEY and not ZHIPU_API_KEY.startswith('your-'))}")
print(f"聊天模型: {CHAT_MODEL}")
print(f"Embedding 模型: {EMBEDDING_MODEL}")

智谱 API Key 是否已设置: True
聊天模型: glm-4.6v-flashx
Embedding 模型: embedding-3


## 1. 导入依赖并定义智谱适配器

In [ ]:
# 可选：在当前 kernel 环境中安装依赖
# !uv pip install langchain langchain-community langchain-text-splitters faiss-cpu zhipuai pyjwt==2.8.0 cachetools

In [2]:
from typing import List

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.embeddings import Embeddings
from zhipuai import ZhipuAI


class ZhipuAIEmbeddings(Embeddings):
    def __init__(self, api_key: str, model: str = "embedding-3"):
        self.client = ZhipuAI(api_key=api_key)
        self.model = model

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # 智谱 embedding 接口单次最多支持 64 条输入，分批处理
        batch_size = 64
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            resp = self.client.embeddings.create(model=self.model, input=batch)
            all_embeddings.extend([item.embedding for item in resp.data])
        return all_embeddings

    def embed_query(self, text: str) -> List[float]:
        return self.embed_documents([text])[0]


class ChatZhipuAI(BaseChatModel):
    api_key: str
    model: str = "glm-4-flash"
    temperature: float = 0.7
    max_tokens: int = 1024

    def __init__(self, api_key: str, model: str = "glm-4-flash", temperature: float = 0.7, max_tokens: int = 1024, **kwargs):
        super().__init__(api_key=api_key, model=model, temperature=temperature, max_tokens=max_tokens, **kwargs)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        client = ZhipuAI(api_key=self.api_key)
        zhipu_messages = []
        for m in messages:
            if isinstance(m, SystemMessage):
                zhipu_messages.append({"role": "system", "content": m.content})
            elif isinstance(m, HumanMessage):
                zhipu_messages.append({"role": "user", "content": m.content})
            elif isinstance(m, AIMessage):
                zhipu_messages.append({"role": "assistant", "content": m.content or ""})
        response = client.chat.completions.create(
            model=self.model,
            messages=zhipu_messages,
            temperature=self.temperature,
            max_tokens=self.max_tokens,
        )
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=response.choices[0].message.content))])

    def _llm_type(self) -> str:
        return "zhipuai"

print("依赖导入完成，智谱适配器已定义")

/tmp/ipykernel_24952/2577238635.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


依赖导入完成，智谱适配器已定义


## 2. 加载与切分文档

In [3]:
novel_path = "/root/rag-agent-learning/【南锦】洛烟小姐的脚下埋着尸体⊙103.7万字（完本）.txt"

with open(novel_path, "r", encoding="utf-8") as f:
    novel_text = f.read()

print(f"小说总字数: {len(novel_text)}")
print(f"小说总行数: {novel_text.count(chr(10))}")

# 只取前 10 万字进行演示，避免 Embedding 调用时间过长和费用过高
demo_text = novel_text[:100000]


def clean_text(text: str) -> str:
    """清洗小说文本：去掉整理声明、乱码符号和多余空行。"""
    import re
    # 去掉常见的整理声明段落
    text = re.sub(r"本书由【.*?】整理.*?及时删除。", "", text, flags=re.DOTALL)
    # 去掉连续 3 个及以上的无意义符号
    text = re.sub(r"[\\|%&$#@_]{3,}", "", text)
    # 多个空行合并成两个换行
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    return text.strip()


def split_by_chapters(text: str):
    """按章节标题拆分小说。"""
    import re
    # 匹配：第一章、第1章、第 一 章 等
    pattern = r"(第[一二三四五六七八九十百千万\d]+章[^\n]*)"
    parts = re.split(pattern, text)

    chapters = []
    for i in range(1, len(parts), 2):
        title = parts[i].strip()
        content = parts[i + 1].strip() if i + 1 < len(parts) else ""
        if title and content:
            chapters.append({"title": title, "content": content})
    return chapters


# 清洗文本
cleaned_text = clean_text(demo_text)
print(f"\n清洗后字数: {len(cleaned_text)}")

# 按章节拆分
chapters = split_by_chapters(cleaned_text)
print(f"识别到章节数量: {len(chapters)}")
for c in chapters[:3]:
    print(f"  - {c['title']}: {len(c['content'])} 字")

# 配置细切分器：优先按段落、对话结束、句子切分
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", "”", "。", "！", "？", " ", ""]
)

# 章节内细切，短章直接作为一块，长章再切
all_chunks = []
for chapter in chapters:
    header = chapter["title"]
    content = chapter["content"]

    if len(content) <= 800:
        # 短章直接保留
        all_chunks.append(Document(
            page_content=f"{header}\n\n{content}",
            metadata={"chapter": header, "source": novel_path}
        ))
    else:
        # 长章再细切
        docs = [Document(page_content=content, metadata={"chapter": header, "source": novel_path})]
        chunks = text_splitter.split_documents(docs)
        for c in chunks:
            c.page_content = f"{header}\n\n{c.page_content}"
            c.metadata["chapter"] = header
        all_chunks.extend(chunks)

print(f"\n最终文档块数量: {len(all_chunks)}")
for i, c in enumerate(all_chunks[:3]):
    print(f"\n--- 块 {i+1} [{c.metadata.get('chapter', '未知章节')}] ---")
    print(c.page_content[:150] + "...")
print("\n...（仅展示前 3 块）")

小说总字数: 1209543
小说总行数: 67911

清洗后字数: 99828
识别到章节数量: 40
  - 第一章 伊始之日: 2414 字
  - 第二章 少女的病: 2353 字
  - 第三章 闯入者: 2509 字

最终文档块数量: 161

--- 块 1 [第一章 伊始之日] ---
第一章 伊始之日

清晨的暖阳准时地穿过窗户落在破旧的木制地板上，将旁边简陋床铺上睡着的银发少女唤醒。

　　少女修长的睫毛抖了抖，缓缓地睁开双眼，动作轻缓地坐了起来。

　　“唔——嗯！”

　　她伸了个懒腰，娇憨的轻吟声令人遐想，青涩的曲线在轻薄的睡衣里若隐若现。

　　“呼......”

　...

--- 块 2 [第一章 伊始之日] ---
第一章 伊始之日

——不过，她现在还活着，这已经是最好的消息了。

　　洛烟对她现在的生活很满意，即使她患有重病，却还是有很多人向她伸出了援手，愿意不求回报地帮助她。

　　小镇上的医院免费为她看病、体检，还时常过来关心宽慰，洛烟心底都是满满的感动。

　　而且，似乎是怕她无法心安理得地接受如此多...

--- 块 3 [第一章 伊始之日] ---
第一章 伊始之日

金黄色的淡薄色彩洒下，穿过浓浓的灰色雾气，为这副充满幸福气息的画面增添了几笔温暖的颜色。

　　洛烟稍稍仰起头，迎着天穹上为大地洒落阳光的太阳，眼角勾出了弯弯的月牙，

　　“阳光真好。”

　　然后，她撑开伞，走出门外，娇小的身躯躲进了被光芒包围的阴影里。

　　即使不知道这样...

...（仅展示前 3 块）


## 3. 配置 Embedding 模型与 FAISS 向量存储

In [4]:
embedding_model = ZhipuAIEmbeddings(api_key=ZHIPU_API_KEY, model=EMBEDDING_MODEL)
vectorstore = FAISS.from_documents(all_chunks, embedding_model)

print("FAISS 向量数据库构建完成")
print(f"向量维度: {vectorstore.index.d}")
print(f"索引中文本块数量: {vectorstore.index.ntotal}")

FAISS 向量数据库构建完成
向量维度: 2048
索引中文本块数量: 161


## 4. 创建检索器并测试检索

In [11]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

query = "洛烟小姐是谁？"
retrieved_docs = retriever.invoke(query)

retrieved_text = "\n\n".join([d.page_content for d in retrieved_docs])
print(f"查询: {query}")
print(f"召回结果数量: {len(retrieved_docs)}")
print(f"召回内容总字符数: {len(retrieved_text)}\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- 结果 {i+1} [{doc.metadata.get('chapter', '未知章节')}] ---")
    print(doc.page_content)
    print(doc.metadata, "\n")

查询: 洛烟小姐是谁？
召回结果数量: 5
召回内容总字符数: 2376

--- 结果 1 [第十三章  “苍白”] ---
第十三章  “苍白”

所以在花绮玲真的回应她，而且看起来非常友好的时候，洛烟明显有些措手不及。

　　“没有。”花绮玲用“鹰眼”稍稍打量着洛烟，温和的表情没有任何变化，“请问有什么事情吗？”

　　——出乎意料。

　　真正直面这位困扰她多时的银发少女，花绮玲忽然发现对方...似乎有些不正常。

　　不...应该说，“苍白”很正常，非常正常。

　　除去绝美到非人的容颜与外貌以外，“苍白”与炎城里随处可见的普通女孩没什么两样。

　　但是，这种正常出现在“毁”级灾厄污染地区“幽灵小镇”里，就是最不正常的地方。

　　当花绮玲转过身来的时候，洛烟才注意到花绮玲眼上蒙着的白布条，发出了一声小小的低呼，“啊，这位小姐，你的眼睛...”

　　“一些旧伤而已，不必在意。”

　　花绮玲镇静地说道，在初见时的些许惊慌过后，她已经逐渐冷静下来了。

　　“原来是这样啊，好吧...”

　　洛烟的心底闪过几分怜悯，她不过是看不清大部分东西就已经很难受了，而这位姐姐却什么看不到...

　　这样的念头在洛烟心底一闪而逝，然后她便礼貌地提起裙角行了一个女仆礼，

　　“先做个自我介绍吧，我叫洛烟，是娜塔莉夫人家的帮佣，欢迎你来到小镇。”

　　花绮玲见状，眼神瞬间锐利了不少，

　　“这是，大陆西方风格的礼仪...”

　　世界上存在着许多巨型城市，各个巨型城市都有其独特的风俗习惯。

　　其中有些是来源于旧纪元，洛烟所行的提裙礼便是其中之一，是大陆西方遗留下来的。

　　但是...“洛烟”，听起来却更像是大陆东方风格的名字。

　　或许从这些支离破碎的信息里，能够探索出幽灵小镇真正的模样。

　　花绮玲记下了这个名字，然后点点头道，“我的名字是花绮玲。”

　　“很好听的名字，”洛烟浅笑着说道，“其实也没什么啦，难得遇到外来的客人，所以有些好奇...花绮玲小姐，你是从哪里来的？”
{'chapter': '第十三章  “苍白”', 'source': '/root/rag-agent-learning/【南锦】洛烟小姐的脚下埋着尸体⊙103.7万字（完本）.txt'} 

--- 结果 2 [第三十八章 画册与读书会] ---
第三十八章 画册与读书会


## 5. 配置 LLM 并运行 RAG 问答

In [12]:
llm = ChatZhipuAI(
    api_key=ZHIPU_API_KEY,
    model=CHAT_MODEL,
    temperature=0.0,
    max_tokens=1024
)
print("LLM 初始化完成")
print(f"使用模型: {llm.model}")


def rag_answer(question: str, top_k: int = 5) -> str:
    """基于 FAISS 检索 + 智谱 LLM 生成答案"""
    retrieved = retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in retrieved])
    context_tokens = len(context)
    print(f"[RAG] 召回 {len(retrieved)} 条参考资料，合并后约 {context_tokens} 个字符")
    prompt = f"""请根据以下参考资料回答问题。如果资料中找不到答案，请说明无法回答。

参考资料：
{context}

问题：{question}
"""
    response = llm.invoke(prompt)
    return response.content


# 测试 RAG 问答
rag_query = "小说的女主角有什么特点？"
answer = rag_answer(rag_query)

print("\n" + "=" * 40)
print("问题：", rag_query)
print("=" * 40)
print("答案：")
print(answer)

LLM 初始化完成
使用模型: glm-4.6v-flashx
[RAG] 召回 5 条参考资料，合并后约 1606 个字符

问题： 小说的女主角有什么特点？
答案：

根据提供的参考资料，小说女主角洛烟的特点包括：  
1. **观察力敏锐**：能察觉仆人变得陌生，通过灵魂气息判断出眼前人物是小镇居民，还能感知环境变化（如“正在趋于她所认知的娜塔莉庄园”）；  
2. **行动力强、果断**：在剧情偏离原剧本时表现出焦急，主动要求“突围”，且目标明确（如专注寻找小蒂妮的画册）；  
3. **有反思与醒悟能力**：从对小镇的迷茫到幡然醒悟，理解了“你拯救了我们”的真相，视野中的“涂鸦视野”被彻底撕裂；  
4. **对他人有善意/担当**：曾救过娜塔莉夫人，对他人（如娜塔莉夫人、外来者）表现出善意，即使外来者认为小镇危险，她仍能反思小镇的本质；  
5. **性格中带有冷静/理性特质**：面对小蒂妮的靠近时，未像以前那样主动拥抱，而是先观察娜塔莉夫人，显示其冷静的处理方式。


## 6. 更多 RAG 问答示例

In [15]:
questions = [
    "洛烟小姐的是否身体健康？",
    "幽灵小镇藏着什么秘密？",
    "文中出现了几个势力"
]

for q in questions:
    print(f"\n问题：{q}")
    print("-" * 40)
    print(rag_answer(q))


问题：洛烟小姐的是否身体健康？
----------------------------------------
[RAG] 召回 5 条参考资料，合并后约 3972 个字符

根据参考资料，洛烟小姐身体健康吗？  

资料中明确提到她是“一名重症病患”，患有类似白化病的疾病，症状包括“无法长时间经受阳光的照耀，体质虚弱，更重要的是，她身上各种器官的机能都在不断持续地衰竭，无时无刻地感受着病痛的折磨”，且“身体就这样一天不如一天，逐渐变得虚弱”。因此，洛烟小姐身体不健康。  

答案：不健康，她是重症病患，身体虚弱且器官机能衰竭。

问题：幽灵小镇藏着什么秘密？
----------------------------------------
[RAG] 召回 5 条参考资料，合并后约 2957 个字符

根据参考资料，幽灵小镇藏着以下秘密：  
1. 会进行不规律移动，曾与巨型城市重叠并被直接同化吞噬；  
2. 目前所有档案资料及探索人员在24小时内的通讯均未提到小镇里存在生命体；  
3. “破晓”组织的干部（能力为短距离空间跃迁）曾逃入小镇，且“湮灭”这一概念与小镇相关联，暗示小镇可能影响外界；  
4. 小镇内部存在某种力场，可隔绝通讯脉冲（如花绮玲的通讯设备无法链接）。

问题：文中出现了几个势力
----------------------------------------
[RAG] 召回 5 条参考资料，合并后约 3575 个字符

